# Pandas from First Principles
## Notebook 12: Performance & Best Practices (Final Notebook)

---

**What you already know:** All major Pandas concepts — Series, DataFrame, cleaning, strings, sorting, apply, groupby, merge, concat, reshape, I/O.

**What you will learn here:** How to use Pandas WELL — faster code, less memory, cleaner style.

---

### Table of Contents

1. Introduction
2. Memory Optimization
3. Vectorization vs apply()
4. Method Chaining
5. copy() and SettingWithCopyWarning
6. Common Beginner Mistakes
7. Practice Questions
8. Complete Series Cheat Sheet
9. Final Interview Notes

---

# 1. Introduction

You have learned all the tools. Now learn how to use them **well**.

Three things that separate a beginner from a professional:

| | Beginner | Professional |
|---|---|---|
| **Speed** | Uses `apply()` for everything | Uses vectorized operations |
| **Memory** | Loads full file, all columns | Selects columns, optimizes dtypes |
| **Readability** | Separate steps, unclear variable names | Method chaining, clean pipeline |

This notebook teaches you those differences.

In [ ]:
import pandas as pd
import numpy as np
import time

print("Pandas version:", pd.__version__)

---

# 2. Memory Optimization

## Why it matters

A 500MB CSV file can easily become a 2–3GB DataFrame in memory because Pandas uses int64 and float64 by default — even when int8 or int16 would be enough. Running out of memory crashes your kernel. Reducing memory usage lets you work with larger data.

## 2.1 Checking memory usage

In [ ]:
# Create a large DataFrame to demonstrate memory
np.random.seed(42)
n = 100_000

big_df = pd.DataFrame({
    "user_id":    np.random.randint(1, 10000, n),
    "age":        np.random.randint(18, 70, n),
    "score":      np.random.randint(0, 100, n),
    "revenue":    np.random.uniform(100, 50000, n).round(2),
    "city":       np.random.choice(["Delhi", "Mumbai", "Pune", "Bangalore", "Chennai"], n),
    "department": np.random.choice(["Eng", "Marketing", "HR", "Finance"], n),
    "is_active":  np.random.choice([True, False], n),
})

print("Shape:", big_df.shape)
print("\nMemory usage per column (in KB):")
print((big_df.memory_usage(deep=True) / 1024).round(1))
print(f"\nTotal memory: {big_df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

## 2.2 Downcasting numeric types

Pandas uses `int64` (8 bytes) by default. But if your values fit in `int8` (range: -128 to 127) or `int16` (range: -32768 to 32767), you are wasting memory.

| dtype | Bytes | Range |
|---|---|---|
| int8 | 1 | -128 to 127 |
| int16 | 2 | -32768 to 32767 |
| int32 | 4 | ~-2 billion to 2 billion |
| int64 | 8 | very large range |

In [ ]:
# Downcast numeric columns to the smallest type that fits the values
df_optimized = big_df.copy()

# age is 18–70 → fits in int8 (or int16 to be safe)
df_optimized["age"] = pd.to_numeric(df_optimized["age"], downcast="integer")

# score is 0–100 → fits in int8
df_optimized["score"] = pd.to_numeric(df_optimized["score"], downcast="integer")

# revenue is float → downcast to float32
df_optimized["revenue"] = pd.to_numeric(df_optimized["revenue"], downcast="float")

print("Optimized dtypes:")
print(df_optimized.dtypes)
print(f"\nBefore: {big_df.memory_usage(deep=True).sum()/1024/1024:.2f} MB")
print(f"After:  {df_optimized.memory_usage(deep=True).sum()/1024/1024:.2f} MB")

## 2.3 Category dtype for string columns with low cardinality

**Low cardinality** means the column has very few unique values compared to the total rows. For example, a `city` column with 5 unique values in 100,000 rows.

Instead of storing the string `"Delhi"` 20,000 times, `category` dtype stores it once and uses small integers internally.

In [ ]:
# Memory before
city_mem_before = big_df["city"].memory_usage(deep=True)
dept_mem_before = big_df["department"].memory_usage(deep=True)

# Convert to category
df_optimized["city"] = df_optimized["city"].astype("category")
df_optimized["department"] = df_optimized["department"].astype("category")

# Memory after
city_mem_after = df_optimized["city"].memory_usage(deep=True)
dept_mem_after = df_optimized["department"].memory_usage(deep=True)

print(f"city memory:       {city_mem_before/1024:.1f} KB → {city_mem_after/1024:.1f} KB")
print(f"department memory: {dept_mem_before/1024:.1f} KB → {dept_mem_after/1024:.1f} KB")

print(f"\nFinal total: {df_optimized.memory_usage(deep=True).sum()/1024/1024:.2f} MB")
print(f"Original:    {big_df.memory_usage(deep=True).sum()/1024/1024:.2f} MB")

**Rule of thumb:** Use `category` dtype when a string column has fewer than 50 unique values or when unique values are less than 5% of total rows.

---

# 3. Vectorization vs apply()

## What is vectorization?

Vectorized operations work on the entire column at once using C-level code. `apply()` runs a Python function row by row in a loop. Python loops are 10x–100x slower.

**Rule:** Always try vectorized first. Use `apply()` only when no vectorized alternative exists.

In [ ]:
# Comparing speed: apply() vs vectorized operation
sample = big_df[["revenue", "score"]].copy()

# Method 1: apply() — slow
start = time.time()
result_apply = sample["revenue"].apply(lambda x: x * 1.18)  # add 18% tax
time_apply = time.time() - start

# Method 2: vectorized — fast
start = time.time()
result_vectorized = sample["revenue"] * 1.18
time_vectorized = time.time() - start

print(f"apply():     {time_apply*1000:.2f} ms")
print(f"vectorized:  {time_vectorized*1000:.2f} ms")
print(f"Speedup:     {time_apply/time_vectorized:.0f}x faster")

## 3.1 Common operations — vectorized alternatives

In [ ]:
df = pd.DataFrame({
    "name":   ["Alice", "Bob", "Carol", "Dave"],
    "salary": [85000, 72000, 91000, 68000],
    "age":    [28, 34, 26, 41],
})

# ❌ SLOW — using apply for simple math
df["bonus_slow"] = df["salary"].apply(lambda x: x * 0.10)

# ✅ FAST — vectorized
df["bonus_fast"] = df["salary"] * 0.10

print("Both produce same result:")
print(df[["name", "salary", "bonus_slow", "bonus_fast"]])

In [ ]:
# ❌ SLOW — apply for conditional
df["level_slow"] = df["salary"].apply(lambda x: "Senior" if x > 80000 else "Junior")

# ✅ FAST — np.where
df["level_fast"] = np.where(df["salary"] > 80000, "Senior", "Junior")

print(df[["name", "salary", "level_slow", "level_fast"]])

In [ ]:
# ❌ SLOW — apply for string operations
df["name_slow"] = df["name"].apply(lambda x: x.upper())

# ✅ FAST — .str accessor
df["name_fast"] = df["name"].str.upper()

print(df[["name", "name_slow", "name_fast"]])

**When IS apply() the right choice?**

- Complex custom logic that cannot be expressed as arithmetic or string ops
- Calling an external function on each row
- When you need to access multiple columns in one function (axis=1)

---

# 4. Method Chaining

## What is it?

Method chaining means calling multiple methods one after another on the same object, without creating intermediate variables.

It makes your data pipeline **readable** — you can read it top to bottom like a recipe.

In [ ]:
# Data for the example
raw_data = pd.DataFrame({
    "Name":       ["  alice ", "BOB", "carol", "  DAVE  ", "eve", None],
    "salary":     ["85000", "72000", "91000", "68000", None, "75000"],
    "department": ["engineering", "MARKETING", "engineering", "HR", "marketing", "HR"],
    "age":        [28, 34, 26, 41, 30, 35],
})

print("Raw data:")
print(raw_data)

In [ ]:
# WITHOUT chaining — many intermediate variables, hard to read
step1 = raw_data.dropna(subset=["Name"])
step2 = step1.rename(columns={"Name": "name"})
step3 = step2.copy()
step3["name"] = step3["name"].str.strip().str.title()
step3["salary"] = pd.to_numeric(step3["salary"], errors="coerce").fillna(0)
step3["department"] = step3["department"].str.title()
step4 = step3.sort_values("salary", ascending=False).reset_index(drop=True)

print("Without chaining (many steps):")
print(step4)

In [ ]:
# WITH chaining — same result, much cleaner
result = (
    raw_data
    .dropna(subset=["Name"])                                         # remove rows with no name
    .rename(columns={"Name": "name"})                                # lowercase column name
    .assign(
        name       = lambda df: df["name"].str.strip().str.title(),  # clean name
        salary     = lambda df: pd.to_numeric(df["salary"], errors="coerce").fillna(0),
        department = lambda df: df["department"].str.title(),
    )
    .sort_values("salary", ascending=False)
    .reset_index(drop=True)
)

print("With chaining (clean pipeline):")
print(result)

## 4.1 Tips for method chaining

- **Wrap in parentheses `()`** to break across multiple lines — cleaner than `\`
- **Use `.assign()`** to add columns in a chain (instead of `df['col'] = ...` which breaks the chain)
- **Use `lambda df:`** inside `.assign()` to reference the DataFrame being built
- **Debug tip:** When a chain is not working, split it into steps temporarily, find the bug, then chain again

## 4.2 .pipe() — custom function in a chain

When you need to apply a custom function in the middle of a chain, use `.pipe()`.

In [ ]:
# Custom function for use in a chain
def add_seniority(df, threshold=32):
    df["is_senior"] = df["age"] > threshold
    return df

# Using .pipe() to call it in a chain
final = (
    result
    .pipe(add_seniority, threshold=32)   # custom function injected into chain
    .query("salary > 0")                  # filter using query string
)

print(final)

---

# 5. copy() and SettingWithCopyWarning

## What is the warning?

```
SettingWithCopyWarning: A value is trying to be set on a copy of a slice from a DataFrame.
```

This warning appears when you try to modify a subset of a DataFrame that Pandas considers a **view** (not a copy). Your modification might silently fail.

## Why it happens

In [ ]:
employees = pd.DataFrame({
    "name":   ["Alice", "Bob", "Carol", "Dave"],
    "dept":   ["Eng", "Marketing", "Eng", "HR"],
    "salary": [85000, 72000, 91000, 68000],
})

# ❌ PROBLEM: chained indexing on a slice
# This may or may not modify the original — behavior is undefined
eng_subset = employees[employees["dept"] == "Eng"]
# eng_subset["salary"] = 100000   # ← This would cause SettingWithCopyWarning

print("Subset (might be a view, might be a copy — undefined!):")
print(eng_subset)

In [ ]:
# ✅ FIX 1: use .copy() when you plan to modify the subset
eng_subset = employees[employees["dept"] == "Eng"].copy()
eng_subset["salary"] = 100000   # Now safe — modifying a true copy

print("Modified copy (original unchanged):")
print(eng_subset)
print("\nOriginal employees (unchanged):")
print(employees)

In [ ]:
# ✅ FIX 2: use .loc to modify the original directly (no copy needed)
employees.loc[employees["dept"] == "Eng", "salary"] = 100000

print("Original modified in place using .loc:")
print(employees)

**Simple rule to remember:**

- If you want to **modify a subset** of a DataFrame → use `.copy()` first
- If you want to **modify the original** DataFrame conditionally → use `.loc[condition, column] = value`

---

# 6. Common Beginner Mistakes

A consolidated list of the most common mistakes across the entire series.

In [ ]:
df = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})

# ❌ MISTAKE 1: Forgetting that most operations return new objects
df.sort_values("a", ascending=False)  # result is discarded!
print("Mistake 1 — df is unchanged:", df["a"].tolist())

# ✅ CORRECT
df = df.sort_values("a", ascending=False).reset_index(drop=True)
print("Correct — reassigned:", df["a"].tolist())

In [ ]:
# ❌ MISTAKE 2: Using Python 'and'/'or' instead of '&'/'|'
df2 = pd.DataFrame({"x": [1, 2, 3, 4, 5], "y": [10, 20, 30, 40, 50]})

try:
    result = df2[(df2["x"] > 2) and (df2["y"] < 45)]  # ValueError!
except ValueError as e:
    print("Mistake 2 error:", type(e).__name__)

# ✅ CORRECT — use & and wrap each condition in ()
result = df2[(df2["x"] > 2) & (df2["y"] < 45)]
print("Correct result:\n", result)

In [ ]:
# ❌ MISTAKE 3: Not resetting index after filter/concat
df3 = pd.DataFrame({"val": [10, 20, 30, 40, 50]})
filtered = df3[df3["val"] > 15]
print("After filter (index is messy):", filtered.index.tolist())

# ✅ CORRECT
filtered = filtered.reset_index(drop=True)
print("After reset_index:", filtered.index.tolist())

In [ ]:
# ❌ MISTAKE 4: Calling .astype(int) without filling NaN first
s = pd.Series([1.0, 2.0, None, 4.0])
try:
    s.astype(int)  # fails!
except Exception as e:
    print("Mistake 4 error:", type(e).__name__, str(e)[:50])

# ✅ CORRECT
s_fixed = s.fillna(0).astype(int)
print("Correct:", s_fixed.tolist())

In [ ]:
# ❌ MISTAKE 5: Forgetting index=False when writing CSV
import io

df_out = pd.DataFrame({"name": ["Alice", "Bob"], "salary": [85000, 72000]})

# With index=True (default) — creates ugly unnamed column on re-read
buf = io.StringIO()
df_out.to_csv(buf, index=True)
buf.seek(0)
df_read_bad = pd.read_csv(buf)
print("With index=True (extra column!):")
print(df_read_bad)

# With index=False — clean
buf2 = io.StringIO()
df_out.to_csv(buf2, index=False)
buf2.seek(0)
df_read_good = pd.read_csv(buf2)
print("\nWith index=False (clean):")
print(df_read_good)

---

# 7. Practice Questions

## Level 1 — Easy

### Q1 — Memory Check

Using the DataFrame below:
- a) Check memory usage per column
- b) Which column uses the most memory?
- c) Convert `city` to category dtype
- d) Compare memory before and after

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(1)
q1_df = pd.DataFrame({
    "id":     np.arange(50000),
    "age":    np.random.randint(18, 65, 50000),
    "salary": np.random.randint(30000, 200000, 50000),
    "city":   np.random.choice(["Delhi", "Mumbai", "Pune"], 50000),
    "dept":   np.random.choice(["Eng", "HR", "Marketing"], 50000),
})

# Q1 — Your solution here


### Q2 — Vectorize it

Rewrite the following `apply()` calls as vectorized operations:

```python
df["tax"]     = df["salary"].apply(lambda x: x * 0.30)
df["is_high"] = df["salary"].apply(lambda x: True if x > 100000 else False)
df["name_up"] = df["name"].apply(lambda x: x.upper())
```

In [ ]:
q2_df = pd.DataFrame({
    "name":   ["Alice", "Bob", "Carol"],
    "salary": [85000, 120000, 72000],
})

# Q2 — Rewrite as vectorized (no apply)


---

## Level 2 — Medium

### Q3 — Fix the SettingWithCopyWarning

The code below will cause a `SettingWithCopyWarning`. Fix it properly.

In [ ]:
employees = pd.DataFrame({
    "name":   ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "dept":   ["Eng", "Marketing", "Eng", "HR", "Eng"],
    "salary": [85000, 72000, 91000, 68000, 95000],
})

# BUGGY CODE — causes SettingWithCopyWarning
# eng = employees[employees["dept"] == "Eng"]
# eng["salary"] = eng["salary"] * 1.10   # 10% raise for Eng

# Q3 — Fix the code properly


### Q4 — Method Chaining Pipeline

Rewrite the following 6 separate steps as a single method chain:

```python
step1 = raw.dropna()
step2 = step1.rename(columns={"emp_name": "name", "emp_sal": "salary"})
step3 = step2.copy()
step3["name"] = step3["name"].str.strip().str.title()
step3["salary"] = pd.to_numeric(step3["salary"], errors="coerce")
step4 = step3[step3["salary"] > 60000]
step5 = step4.sort_values("salary", ascending=False).reset_index(drop=True)
```

In [ ]:
raw = pd.DataFrame({
    "emp_name": ["  alice", "BOB ", None, "carol", "  DAVE"],
    "emp_sal":  ["85000", "72000", "91000", "bad", "95000"],
    "dept":     ["Eng", "Marketing", "Eng", "HR", "Eng"],
})

# Q4 — Rewrite as one method chain


### Q5 — Full Optimization

Given the large DataFrame below:
- a) Check total memory
- b) Downcast `age` and `score` to smallest integer type
- c) Convert `region` and `category` to `category` dtype
- d) Downcast `price` to float32
- e) Print memory before and after with percentage saved

In [ ]:
np.random.seed(42)
n = 200_000
q5_df = pd.DataFrame({
    "user_id":  np.random.randint(1, 99999, n),
    "age":      np.random.randint(18, 70, n),
    "score":    np.random.randint(0, 100, n),
    "price":    np.random.uniform(10.0, 5000.0, n),
    "region":   np.random.choice(["North", "South", "East", "West"], n),
    "category": np.random.choice(["A", "B", "C"], n),
})

# Q5 — Your solution here


---

## Answer Key

In [ ]:
# ── Q1 Answer ────────────────────────────────────────────────────────────────
np.random.seed(1)
q1_df = pd.DataFrame({
    "id":     np.arange(50000),
    "age":    np.random.randint(18, 65, 50000),
    "salary": np.random.randint(30000, 200000, 50000),
    "city":   np.random.choice(["Delhi", "Mumbai", "Pune"], 50000),
    "dept":   np.random.choice(["Eng", "HR", "Marketing"], 50000),
})

mem_before = q1_df.memory_usage(deep=True)
print("a) Memory per column (KB):")
print((mem_before / 1024).round(1))
print("b) Most memory:", mem_before.idxmax())

q1_df["city"] = q1_df["city"].astype("category")
q1_df["dept"] = q1_df["dept"].astype("category")
mem_after = q1_df.memory_usage(deep=True)

print(f"\nc+d) Before: {mem_before.sum()/1024:.1f} KB | After: {mem_after.sum()/1024:.1f} KB")

In [ ]:
# ── Q2 Answer ────────────────────────────────────────────────────────────────
q2_df = pd.DataFrame({"name": ["Alice", "Bob", "Carol"], "salary": [85000, 120000, 72000]})

q2_df["tax"]     = q2_df["salary"] * 0.30          # vectorized
q2_df["is_high"] = q2_df["salary"] > 100000         # vectorized boolean
q2_df["name_up"] = q2_df["name"].str.upper()        # .str accessor

print(q2_df)

In [ ]:
# ── Q3 Answer ────────────────────────────────────────────────────────────────
employees = pd.DataFrame({
    "name":   ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "dept":   ["Eng", "Marketing", "Eng", "HR", "Eng"],
    "salary": [85000, 72000, 91000, 68000, 95000],
})

# Fix 1: .copy()
eng = employees[employees["dept"] == "Eng"].copy()
eng["salary"] = eng["salary"] * 1.10
print("Fix 1 (.copy()):")
print(eng)

# Fix 2: .loc on original
employees.loc[employees["dept"] == "Eng", "salary"] = employees.loc[employees["dept"] == "Eng", "salary"] * 1.10
print("\nFix 2 (.loc on original):")
print(employees)

In [ ]:
# ── Q4 Answer ────────────────────────────────────────────────────────────────
raw = pd.DataFrame({
    "emp_name": ["  alice", "BOB ", None, "carol", "  DAVE"],
    "emp_sal":  ["85000", "72000", "91000", "bad", "95000"],
    "dept":     ["Eng", "Marketing", "Eng", "HR", "Eng"],
})

result = (
    raw
    .dropna()
    .rename(columns={"emp_name": "name", "emp_sal": "salary"})
    .assign(
        name   = lambda df: df["name"].str.strip().str.title(),
        salary = lambda df: pd.to_numeric(df["salary"], errors="coerce"),
    )
    .query("salary > 60000")
    .sort_values("salary", ascending=False)
    .reset_index(drop=True)
)

print(result)

In [ ]:
# ── Q5 Answer ────────────────────────────────────────────────────────────────
np.random.seed(42)
n = 200_000
q5_df = pd.DataFrame({
    "user_id":  np.random.randint(1, 99999, n),
    "age":      np.random.randint(18, 70, n),
    "score":    np.random.randint(0, 100, n),
    "price":    np.random.uniform(10.0, 5000.0, n),
    "region":   np.random.choice(["North", "South", "East", "West"], n),
    "category": np.random.choice(["A", "B", "C"], n),
})

before = q5_df.memory_usage(deep=True).sum() / 1024 / 1024

q5_df["age"]      = pd.to_numeric(q5_df["age"], downcast="integer")
q5_df["score"]    = pd.to_numeric(q5_df["score"], downcast="integer")
q5_df["price"]    = pd.to_numeric(q5_df["price"], downcast="float")
q5_df["region"]   = q5_df["region"].astype("category")
q5_df["category"] = q5_df["category"].astype("category")

after = q5_df.memory_usage(deep=True).sum() / 1024 / 1024
saved = (before - after) / before * 100

print(f"Before: {before:.2f} MB")
print(f"After:  {after:.2f} MB")
print(f"Saved:  {saved:.1f}%")

---

# 8. Complete Series Cheat Sheet

## Notebook 1 — Series
| Code | What it does |
|---|---|
| `pd.Series([1,2,3], index=['a','b','c'])` | Create Series with custom index |
| `s.loc['a']` | Access by label |
| `s.iloc[0]` | Access by position |
| `s + other_s` | Aligned arithmetic (NaN where index doesn't match) |
| `s.isna()`, `s.dropna()`, `s.fillna(val)` | Handle missing values |
| `s.value_counts()` | Count unique values |
| `s.str.upper()` | String operations via .str |

## Notebook 2 — DataFrame
| Code | What it does |
|---|---|
| `pd.DataFrame({...})` | Create from dict of lists |
| `df.head()`, `df.info()`, `df.describe()` | Quick inspection |
| `df['col']` → Series | Single column |
| `df[['c1','c2']]` → DataFrame | Multiple columns |
| `df.loc[label]`, `df.iloc[pos]` | Row selection |
| `df[condition]` | Boolean filter |
| `df['new'] = values` | Add column (mutates) |
| `df.drop('col', axis=1)` | Remove column |
| `df.rename(columns={'old':'new'})` | Rename column |

## Notebook 3 — Data Cleaning
| Code | What it does |
|---|---|
| `df.isna().sum()` | Count missing per column |
| `df.dropna(subset=['col'])` | Drop rows where col is NaN |
| `df['col'].fillna(value)` | Fill NaN |
| `df.duplicated()`, `df.drop_duplicates()` | Find/remove duplicates |
| `pd.to_numeric(s, errors='coerce')` | Safe string→numeric |
| `pd.to_datetime(s)` | String→datetime |
| `s.replace({old: new})` | Replace specific values |
| `s.where(condition)` | Keep True, NaN for False |

## Notebook 4 — String & DateTime
| Code | What it does |
|---|---|
| `.str.strip()`, `.str.lower()`, `.str.title()` | Clean string casing |
| `.str.contains('x')` | Boolean: does it contain x? |
| `.str.split('.').str[0]` | Split and extract part |
| `.str.replace('a','b')` | Replace substring |
| `.dt.year`, `.dt.month`, `.dt.day` | Extract date parts |
| `.dt.day_name()` | Day name (Monday etc.) |
| `(date2 - date1).dt.days` | Days between two dates |

## Notebook 5 — Sorting & Ranking
| Code | What it does |
|---|---|
| `df.sort_values('col', ascending=False)` | Sort by column |
| `df.sort_values(['c1','c2'], ascending=[True,False])` | Multi-column sort |
| `df.nlargest(5, 'col')` | Top 5 rows |
| `df.nsmallest(3, 'col')` | Bottom 3 rows |
| `s.rank(method='dense', ascending=False)` | Rank values |

## Notebook 6 — Applying Functions
| Code | What it does |
|---|---|
| `s.apply(func)` | Apply function element-wise |
| `df.apply(func, axis=1)` | Apply row-wise |
| `s.map({old: new})` | Map values using dict |
| `np.where(cond, val_true, val_false)` | Vectorized if-else |
| `pd.cut(s, bins, labels=...)` | Bin into equal-width groups |
| `pd.qcut(s, q, labels=...)` | Bin into equal-frequency groups |

## Notebook 7 — GroupBy
| Code | What it does |
|---|---|
| `df.groupby('col').sum()` | Sum per group |
| `df.groupby('col')['salary'].mean()` | Mean of one column per group |
| `df.groupby(['c1','c2']).agg('sum')` | Multi-column groupby |
| `.agg(['mean','max','count'])` | Multiple aggregations |
| `.agg(avg=('col','mean'))` | Named aggregation |
| `.transform('mean')` | Add group stat back to original shape |

## Notebook 8 — Merge & Join
| Code | What it does |
|---|---|
| `pd.merge(left, right, on='col')` | Inner join (default) |
| `pd.merge(..., how='left')` | Left join |
| `pd.merge(..., how='outer')` | Outer join |
| `pd.merge(l, r, left_on='a', right_on='b')` | Merge on different column names |
| `df.join(other)` | Join on index |

## Notebook 9 — Concat
| Code | What it does |
|---|---|
| `pd.concat([df1, df2])` | Stack rows |
| `pd.concat([df1, df2], ignore_index=True)` | Stack + reset index |
| `pd.concat([df1, df2], axis=1)` | Stack columns side by side |
| `pd.concat([df1, df2], join='inner')` | Keep only common columns |
| `pd.concat([df1, df2], keys=['A','B'])` | Track source with labels |

## Notebook 10 — Reshaping
| Code | What it does |
|---|---|
| `df.pivot_table(values, index, columns, aggfunc)` | Group and reshape to wide |
| `df.melt(id_vars, value_vars)` | Wide → long format |
| `margins=True` in pivot_table | Add row/column totals |

## Notebook 11 — I/O
| Code | What it does |
|---|---|
| `pd.read_csv(path, sep, usecols, dtype, na_values)` | Read CSV |
| `df.to_csv(path, index=False)` | Write CSV |
| `pd.read_excel(path, sheet_name)` | Read Excel |
| `pd.read_json(path, orient='records')` | Read JSON |

## Notebook 12 — Performance
| Code | What it does |
|---|---|
| `df.memory_usage(deep=True)` | Memory per column |
| `pd.to_numeric(s, downcast='integer')` | Downcast to smaller int |
| `s.astype('category')` | Save memory for low-cardinality strings |
| `df.copy()` | True copy before modifying subset |
| `.assign(...).sort_values(...).reset_index()` | Method chaining |
| `.pipe(func)` | Custom function in chain |

---

# 9. Final Interview Notes

## Top 15 Interview Questions (one-line answers)

| Question | Answer |
|---|---|
| What is a DataFrame? | 2D table = dict of Series with shared index |
| `.loc` vs `.iloc`? | `.loc` = by label (inclusive slice), `.iloc` = by position (exclusive slice) |
| `merge` vs `concat`? | `merge` = join on key column, `concat` = stack rows/columns |
| `apply` vs vectorized? | Vectorized = C speed (always prefer). `apply` = Python loop (slower) |
| `pivot_table` vs `pivot`? | `pivot_table` aggregates duplicates, `pivot` requires unique index |
| `dropna` vs `fillna`? | `dropna` removes rows/cols, `fillna` replaces missing with a value |
| What is SettingWithCopyWarning? | Modifying a view instead of a copy. Fix: `.copy()` or `.loc` |
| `inner` vs `left` join? | inner = only matched rows, left = all left rows + matched right |
| `groupby().agg()` vs `transform()`? | `agg` = reduced shape, `transform` = same shape as original |
| Wide vs long format? | Wide = one row per entity, multiple col. Long = one row per measurement |
| What is category dtype? | Efficient string storage for low-cardinality columns |
| How to read only specific columns from CSV? | `pd.read_csv(path, usecols=['c1','c2'])` |
| `pd.cut` vs `pd.qcut`? | `cut` = equal-width bins, `qcut` = equal-frequency (quantile) bins |
| How to handle NaN in astype(int)? | `fillna(0).astype(int)` — always fill first |
| What does `errors='coerce'` do in `pd.to_numeric`? | Converts invalid values to NaN instead of raising error |

## Most Used Functions in Real Projects

1. `pd.read_csv()` — load data
2. `.info()`, `.head()`, `.describe()` — explore
3. `.isna().sum()` — check missing
4. `.fillna()`, `.dropna()` — handle missing
5. `.drop_duplicates()` — remove duplicates
6. `pd.to_numeric()`, `pd.to_datetime()` — fix types
7. `df[condition]` — filter rows
8. `.groupby().agg()` — aggregate
9. `pd.merge()` — combine datasets
10. `.sort_values()` — sort
11. `.rename()`, `.drop()` — clean columns
12. `.assign()` — add columns in chain
13. `np.where()` — conditional columns
14. `.str.strip().str.title()` — clean strings
15. `to_csv(index=False)` — save results

---

## Congratulations!

You have completed the **Pandas from First Principles** series.

**12 notebooks. Every major concept. Built from zero to job-ready.**

| What you can now do |
|---|
| Load and inspect any dataset |
| Clean messy real-world data |
| Filter, sort, rank, and group data |
| Merge and combine multiple DataFrames |
| Reshape data between wide and long formats |
| Read from and write to CSV, Excel, JSON |
| Write clean, efficient, production-quality Pandas code |

**The next step:** Practice on real datasets from Kaggle, UCI ML Repository, or your own work. Every dataset you clean makes the next one faster.